In [ ]:
# NOTEBOOK 30 - real explainability and real uncertainty for the frontend (v2 models, the same 970 playback windows)
# The frontend currently shows hard-coded attention weights, impact bars and a fixed-sigma uncertainty band.
# This notebook computes the real versions from the 7 saved v2 models and writes JSON files into  v2_clean/frontend_export/
# (a new folder; nothing in frontend/ is touched - the frontend developer copies the files into frontend/public/data/).
#   explain_windows.json : per window and station - exact Shapley impact (ug/m3) of six feature groups on the 24-hour mean forecast,
#                          the baseline (all groups at the training mean) and the forecast; plus the real attention over the 48 input hours
#   bands.json           : per window, station and horizon - 10th / 50th / 90th percentile across the 7 seeds (ensemble spread)
#   explain_global.json  : the global results from notebook 22 (Shapley shares, blank-block test)
# It also checks that its ensemble mean reproduces the predictions already in dashboard_export/playback.json.
# Needs: nothing new (uses the saved v2 checkpoints). Run time on the GPU: about 5-10 minutes for all 970 windows.
# HOW TO CHECK: this cell only prints a note.
LIMIT = None          # set to a small number (e.g. 40) for a quick test on a few windows; None = all playback windows
print('frontend export notebook; LIMIT =', LIMIT)

In [ ]:
import os, json, time, copy, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

BASE   = r'D:\FYP\Eco-Twin'
V2_DIR = os.path.join(BASE, 'v2_clean')
OUT    = os.path.join(V2_DIR, 'ecotwin_outputs')
SCALER_DIR, PLOT_DIR   = os.path.join(OUT, 'scalers'), os.path.join(OUT, 'plots')
MODEL_DIR, RESULTS_DIR = os.path.join(OUT, 'models'),  os.path.join(OUT, 'results')

DATASET_CSV      = os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv')
DYNAMIC_ADJ_PATH = os.path.join(V2_DIR, 'dynamic_adj_matrix.npy')
STATIONS_EXCEL   = os.path.join(V2_DIR, 'Delhi_list_v2.xlsx')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

INPUT_WINDOW, HORIZON = 48, 24
BATCH_SIZE = 16
HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, DROPOUT = 32, 1, 4, 0.2
GCN_HOPS       = 2
LOSS_VOL_ALPHA = 1.0                       # volatility up-weighting (0 = plain Huber)
SEEDS          = [42, 7, 123, 2024, 999, 2025, 77]
THRESHOLD_KM   = 12.0
V2_TAG         = 'watgcn_v2'               # 07's files are never overwritten

print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
print(f'Seeds : {SEEDS}')

# HOW TO CHECK: should print 'cuda' + your RTX 3050. 'cpu' still works, ~10x slower.

In [ ]:
df = pd.read_csv(DATASET_CSV, parse_dates=['Timestamp'])
df['Station'] = df['Station'].str.strip()
df = df.sort_values(['Station', 'Timestamp']).reset_index(drop=True)

df_st = pd.read_excel(STATIONS_EXCEL)
df_st.columns = df_st.columns.str.strip().str.lower()
df_st['station'] = df_st['station'].str.strip()
station_names = df_st['station'].tolist()
lats = df_st['latitude'].values.astype(float)
lons = df_st['longitude'].values.astype(float)
N = len(station_names)

timestamps = np.array(sorted(df['Timestamp'].unique()))
T = len(timestamps)
dyn_adj = np.load(DYNAMIC_ADJ_PATH).astype(np.float32)          # (T,N,N) wind A(t), no self-loops
assert dyn_adj.shape == (T, N, N), (dyn_adj.shape, (T, N, N))

print(f'Stations   : {N}')
print(f'Timestamps : {T:,}  ({timestamps[0]} -> {timestamps[-1]})')
print(f'dyn_adj    : {dyn_adj.shape}  {dyn_adj.nbytes/1e6:.0f} MB')

# geometry for the forecast-climatology graph (same recipe as 03_dynamic_adjacency)
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dlmb = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlmb/2)**2
    return 2*R*np.arctan2(np.sqrt(a), np.sqrt(1-a))

dist_matrix = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        if i != j:
            dist_matrix[i, j] = haversine_km(lats[i], lons[i], lats[j], lons[j])
within_threshold = (dist_matrix > 0) & (dist_matrix <= THRESHOLD_KM)
dir_u = np.zeros((N, N)); dir_v = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        if i != j:
            de = (lons[j]-lons[i]) * np.cos(np.radians(lats[i]))
            dn = (lats[j]-lats[i])
            nrm = np.hypot(de, dn)
            if nrm > 0:
                dir_u[i, j], dir_v[i, j] = de/nrm, dn/nrm
print(f'Edges within {THRESHOLD_KM} km: {int(within_threshold.sum())}')

# HOW TO CHECK: 'Edges within 12 km' matches 03_dynamic_adjacency. dyn_adj ~105 MB.

In [ ]:
BASE_FEATS = ['pm2.5','pm10','no2','so2','co','o3','rh','at','u_wind','v_wind']
g = df.groupby('Station', sort=False)

df['wspd'] = np.hypot(df['u_wind'], df['v_wind'])

# rolling PM2.5 stats - window ends at 'now' (an observed hour), so no leakage
df['pm_roll6_mean']  = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).mean())
df['pm_roll6_std']   = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).std()).fillna(0.0)
df['pm_roll24_mean'] = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).mean())
df['pm_roll24_std']  = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).std()).fillna(0.0)
df['pm_delta3']      = g['pm2.5'].transform(lambda s: s - s.shift(3)).fillna(0.0)

tsg = df.groupby('Timestamp', sort=False)['pm2.5']
df['city_mean_pm']  = tsg.transform('mean')
df['city_max_pm']   = tsg.transform('max')
df['pm_minus_city'] = df['pm2.5'] - df['city_mean_pm']

hh  = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month
df['hour_sin'] = np.sin(2*np.pi*hh/24);      df['hour_cos'] = np.cos(2*np.pi*hh/24)
df['dow_sin']  = np.sin(2*np.pi*dow/7);      df['dow_cos']  = np.cos(2*np.pi*dow/7)
df['mon_sin']  = np.sin(2*np.pi*(mon-1)/12); df['mon_cos']  = np.cos(2*np.pi*(mon-1)/12)
df['is_weekend'] = (dow >= 5).astype(float)

FEATURE_COLS = (BASE_FEATS +
    ['wspd','pm_roll6_mean','pm_roll6_std','pm_roll24_mean','pm_roll24_std','pm_delta3',
     'city_mean_pm','city_max_pm','pm_minus_city',
     'hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend'])
F = len(FEATURE_COLS)
TIME_COLS = ['hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend']
TIME_IDX  = [FEATURE_COLS.index(c) for c in TIME_COLS]

X_full = np.zeros((T, N, F), dtype=np.float32)
for fi, col in enumerate(FEATURE_COLS):
    wide = df.pivot_table(index='Timestamp', columns='Station', values=col).reindex(columns=station_names)
    X_full[:, :, fi] = np.nan_to_num(wide.values.astype(np.float32))
Y_full = X_full[:, :, 0].copy()                                  # PM2.5 target (channel 0)

print(f'F = {F} features')
print(FEATURE_COLS)
print(f'X_full {X_full.shape}  {X_full.nbytes/1e6:.0f} MB   NaNs: {int(np.isnan(X_full).sum())}')

# HOW TO CHECK: F == 26, X_full ~77 MB, NaNs == 0.

In [ ]:
train_end_idx = int(T * 0.70)                                    # 13944
val_end_idx   = int(T * 0.85)                                    # 16932

def split_starts(lo, hi):
    n = (hi - lo) - (INPUT_WINDOW + HORIZON) + 1                  # windows fully inside [lo,hi)
    return np.arange(lo, lo + n)

train_starts = split_starts(0, train_end_idx)
val_starts   = split_starts(train_end_idx, val_end_idx)
test_starts  = split_starts(val_end_idx, T)
print(f'windows -- train {len(train_starts):,} | val {len(val_starts):,} | test {len(test_starts):,}')

train_ts_mask = np.zeros(T, bool); train_ts_mask[:train_end_idx] = True

feat_scaler = StandardScaler().fit(X_full[train_ts_mask].reshape(-1, F))
Xs_full = feat_scaler.transform(X_full.reshape(-1, F)).reshape(T, N, F).astype(np.float32)

pm_train   = Y_full[train_ts_mask].reshape(-1, 1)
log_scaler = StandardScaler().fit(np.log1p(np.clip(pm_train, 0, None)))
def to_log(a):
    return log_scaler.transform(np.log1p(np.clip(a, 0, None)).reshape(-1, 1)).reshape(a.shape).astype(np.float32)
def from_log(a):
    return np.expm1(log_scaler.inverse_transform(a.reshape(-1, 1)).reshape(a.shape))
Yl_full = to_log(Y_full)                                         # anchor = Yl_full[now] later

_std = Xs_full[train_ts_mask].reshape(-1, F).std(0)
print(f'raw PM2.5 skew {pd.Series(pm_train.ravel()).skew():.2f} -> log skew '
      f'{pd.Series(np.log1p(pm_train.ravel())).skew():.2f}')
print(f'per-feature std after scaling: min {_std.min():.2f}  max {_std.max():.2f}')

del X_full                                                       # free RAM - only Xs_full needed now

# HOW TO CHECK: window counts 13873 / 2917 / 2918 (same as 07). log skew ~0. std ~1.

In [ ]:
# forecast-horizon adjacency from a TRAIN-only wind climatology
def compute_At_vec(u, v, eps=1e-6):
    align = u[:, None]*dir_u + v[:, None]*dir_v                  # wind at i . direction(i->j)
    A = np.where(within_threshold & (align > 0.0), align/(dist_matrix + eps), 0.0).astype(np.float32)
    rs = A.sum(1, keepdims=True); rs[rs == 0] = 1.0
    return A / rs

trm = df['Timestamp'].isin(pd.DatetimeIndex(timestamps[:train_end_idx]))
cl = (df[trm].assign(mo=df.loc[trm, 'Timestamp'].dt.month - 1,
                     ho=df.loc[trm, 'Timestamp'].dt.hour)
             .groupby(['Station', 'mo', 'ho'])[['u_wind', 'v_wind']].mean())
st_mean = df[trm].groupby('Station')[['u_wind', 'v_wind']].mean()
cl_idx = set(cl.index)

clim_grid = np.zeros((12, 24, N, 2), dtype=np.float32)           # (month, hour, station, uv)
for si, s in enumerate(station_names):
    for mo in range(12):
        for ho in range(24):
            clim_grid[mo, ho, si] = (cl.loc[(s, mo, ho)].values
                                     if (s, mo, ho) in cl_idx else st_mean.loc[s].values)

mo_arr = pd.DatetimeIndex(timestamps).month.values - 1
ho_arr = pd.DatetimeIndex(timestamps).hour.values
t0 = time.time()
dyn_clim = np.zeros((T, N, N), dtype=np.float32)
for t in range(T):
    uv = clim_grid[mo_arr[t], ho_arr[t]]
    dyn_clim[t] = compute_At_vec(uv[:, 0], uv[:, 1])
print(f'dyn_clim {dyn_clim.shape} built in {time.time()-t0:.0f}s  ({dyn_clim.nbytes/1e6:.0f} MB)')
print(f'edges/hr -- hourly A(t) {np.mean((dyn_adj>0).sum((1,2))):.1f} | '
      f'climatology {np.mean((dyn_clim>0).sum((1,2))):.1f}')

import gc; del df, trm, cl; gc.collect()                         # free RAM - df no longer needed

# HOW TO CHECK: climatology edges/hr in the same ballpark as hourly (smoother). <60s.

In [ ]:
class EcoWindowDS(Dataset):
    """Slices the shared full-length arrays by window-start index - no windowed copies."""
    def __init__(self, starts, weights=None):
        self.s = np.asarray(starts, dtype=np.int64)
        self.w = (weights if weights is not None else np.ones(len(self.s), np.float32)).astype(np.float32)
    def __len__(self):
        return len(self.s)
    def __getitem__(self, k):
        gg = int(self.s[k]); i1 = gg + INPUT_WINDOW; i2 = i1 + HORIZON
        return (torch.from_numpy(Xs_full[gg:i1].copy()),           # (48,N,F)
                torch.from_numpy(dyn_adj[gg:i1].copy()),           # (48,N,N)
                torch.from_numpy(dyn_clim[i1:i2].copy()),          # (24,N,N)
                torch.from_numpy(Xs_full[i1:i2, 0, :][:, TIME_IDX].copy()),  # (24,n_time)
                torch.from_numpy(Yl_full[i1-1].copy()),            # (N,) last obs PM2.5 (log)
                torch.from_numpy(Yl_full[i1:i2].copy()),           # (24,N) target (log)
                torch.tensor(self.w[k], dtype=torch.float32))

In [ ]:
class WATGCNv2(nn.Module):
    def __init__(self, in_features, hidden_dim, gru_layers, attn_heads,
                 forecast_horizon, n_stations, dropout=0.2, n_time=7, gcn_hops=2):
        super().__init__()
        H = hidden_dim
        self.input_proj = nn.Linear(in_features, H)
        self.gcn_lin  = nn.ModuleList(nn.Linear(H, H) for _ in range(gcn_hops))
        self.gcn_norm = nn.ModuleList(nn.LayerNorm(H) for _ in range(gcn_hops))
        self.act = nn.ReLU()
        self.gru = nn.GRU(H, H, gru_layers, batch_first=True,
                          dropout=dropout if gru_layers > 1 else 0.0)
        self.attn = nn.MultiheadAttention(H, attn_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(H)
        self.dropout = nn.Dropout(dropout)
        self.time_enc = nn.Sequential(nn.Linear(n_time, H), nn.ReLU(), nn.Linear(H, H))
        self.decoder = nn.Sequential(nn.Linear(H*3, H), nn.ReLU(), nn.Dropout(dropout), nn.Linear(H, 1))
        self.anchor_gate = nn.Parameter(torch.linspace(1.0, 0.3, forecast_horizon))
        self.H, self.FH = H, forecast_horizon

    @staticmethod
    def _norm_adj(A, n, device):
        I = torch.eye(n, device=device).unsqueeze(0).unsqueeze(0)
        A = A + I
        return A / A.sum(-1, keepdim=True).clamp(min=1e-6)

    def forward(self, X, At, Afc, TFut, Anchor):
        B, Tin, n, _ = X.shape
        h = self.act(self.input_proj(X))                          # (B,Tin,n,H)
        A = self._norm_adj(At, n, X.device)
        for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
            h = self.act(nrm(lin(torch.matmul(A, h)))) + h        # residual graph hop
        hf = h.permute(0, 2, 1, 3).reshape(B*n, Tin, self.H)
        gg, _ = self.gru(hf)
        gg = self.dropout(gg)
        aa, _ = self.attn(gg, gg, gg)
        ctx = self.attn_norm(aa + gg)[:, -1, :].reshape(B, n, self.H)   # (B,n,H)
        Afc_n = self._norm_adj(Afc, n, X.device)                  # (B,FH,n,n)
        ctx_fc = torch.matmul(Afc_n, ctx.unsqueeze(1).expand(B, self.FH, n, self.H))  # (B,FH,n,H)
        t_emb = self.time_enc(TFut).unsqueeze(2).expand(B, self.FH, n, self.H)
        ctx_rep = ctx.unsqueeze(1).expand(B, self.FH, n, self.H)
        resid = self.decoder(torch.cat([ctx_rep, ctx_fc, t_emb], dim=-1)).squeeze(-1)  # (B,FH,n)
        anchor = Anchor.unsqueeze(1) * self.anchor_gate.view(1, self.FH, 1)
        return resid + anchor

_m = WATGCNv2(F, HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, HORIZON, N, DROPOUT, len(TIME_IDX), GCN_HOPS)
print(f'v2 parameters: {sum(p.numel() for p in _m.parameters()):,}')
del _m

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval(); P, A = [], []
    for X, At, Afc, tf, anc, y, w in loader:
        out = model(X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE), tf.to(DEVICE), anc.to(DEVICE))
        P.append(out.cpu().numpy()); A.append(y.numpy())
    return np.concatenate(P), np.concatenate(A)

def score(pred_log, act_log):
    pr, ar = from_log(pred_log), from_log(act_log)
    return dict(rmse=float(np.sqrt(mean_squared_error(ar.ravel(), pr.ravel()))),
                mae=float(mean_absolute_error(ar.ravel(), pr.ravel())),
                r2=float(r2_score(ar.ravel(), pr.ravel())), preds=pr, actual=ar)

In [ ]:
# load the 7 v2 models and the playback windows that the frontend already shows
models = []
for sd in SEEDS:
    m = WATGCNv2(F, HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, HORIZON, N, DROPOUT, len(TIME_IDX), GCN_HOPS).to(DEVICE)
    m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'watgcn_v2_seed{sd}.pt'), map_location=DEVICE, weights_only=True)); m.eval(); models.append(m)
PB = json.load(open(os.path.join(V2_DIR, 'dashboard_export', 'playback.json')))
widx = [int(f['window_idx']) for f in PB]
if LIMIT: widx = widx[:LIMIT]; PB = PB[:LIMIT]
print(f'{len(models)} models | {len(widx)} windows | test windows available {len(test_starts)}')
ds = EcoWindowDS(test_starts)

groups = {
    'PM2.5 history':   ['pm2.5','pm_roll6_mean','pm_roll6_std','pm_roll24_mean','pm_roll24_std','pm_delta3'],
    'Other pollutants':['pm10','no2','so2','co','o3'],
    'Weather (T, RH)': ['rh','at'],
    'Wind':            ['u_wind','v_wind','wspd'],
    'City context':    ['city_mean_pm','city_max_pm','pm_minus_city'],
    'Calendar (input)':['hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend'],
}
gnames = list(groups); n_g = len(gnames)
gidx = {g: [FEATURE_COLS.index(c) for c in cols] for g, cols in groups.items()}
assert sum(len(v) for v in gidx.values()) == F
import itertools, math
# Shapley weights: phi_g = sum over coalitions S without g of |S|!(n-|S|-1)!/n! * (v(S+g) - v(S));  S is a bitmask
SHAP_TERMS = []
for g in range(n_g):
    others = [x for x in range(n_g) if x != g]
    for r in range(len(others) + 1):
        for S in itertools.combinations(others, r):
            mask = sum(1 << x for x in S)
            SHAP_TERMS.append((g, mask, mask | (1 << g), math.factorial(r) * math.factorial(n_g - r - 1) / math.factorial(n_g)))
print('coalitions:', 2 ** n_g, '| shapley terms:', len(SHAP_TERMS))

In [ ]:
# main loop: for each chunk of windows run all 64 coalitions through the 7 models; keep per-seed predictions for the full model
@torch.no_grad()
def run_models(X, At, Afc, TF, Anc, capture_attn=False):
    outs, attn = [], []
    for m in models:
        store = []
        hk = m.attn.register_forward_hook(lambda mod, inp, out: store.append(out[1].detach().cpu())) if capture_attn else None
        outs.append(m(X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE), TF.to(DEVICE), Anc.to(DEVICE)).cpu().numpy())
        if hk is not None:
            hk.remove(); attn.append(store[0][:, -1, :].numpy())                          # (B*n, 48): last position over the input hours
    return np.stack(outs), (np.mean(attn, axis=0) if capture_attn else None)             # (7,B,24,n) log preds, (B*n,48)

CH = 40
phi_all, base_all, fc_all, attn_all, band_all = [], [], [], [], []
t0 = time.time()
for c0 in range(0, len(widx), CH):
    items = [ds[k] for k in widx[c0:c0 + CH]]
    X, At, Afc, TF, Anc, Y, _w = [torch.stack([it[j] for it in items]) for j in range(7)]
    B = X.size(0)
    v = np.zeros((2 ** n_g, B, N), dtype=np.float32)                                      # mean 24-hour forecast per coalition
    for mask in range(2 ** n_g):
        Xm = X.clone()
        for g in range(n_g):
            if not (mask >> g) & 1: Xm[:, :, :, gidx[gnames[g]]] = 0.0                      # absent group = training mean (0 after scaling)
        outs, attn = run_models(Xm, At, Afc, TF, Anc, capture_attn=(mask == 2 ** n_g - 1))
        pred = from_log(np.mean(outs, axis=0))                                             # ensemble mean in ug/m3, (B,24,n)
        v[mask] = pred.mean(axis=1)
        if mask == 2 ** n_g - 1:
            seeds_pred = from_log(outs)                                                    # (7,B,24,n)
            full_pred = pred
            attn_all.append(attn.reshape(B, N, INPUT_WINDOW))
    phi = np.zeros((B, N, n_g), dtype=np.float32)
    for g, m_without, m_with, w in SHAP_TERMS:
        phi[:, :, g] += w * (v[m_with] - v[m_without])
    phi_all.append(phi); base_all.append(v[0]); fc_all.append(v[2 ** n_g - 1])
    band_all.append(np.stack([np.percentile(seeds_pred, 10, axis=0), np.median(seeds_pred, axis=0), np.percentile(seeds_pred, 90, axis=0)]))   # (3,B,24,n)
    chk = np.abs(full_pred - np.array([[[s['predicted_24h'][h] for s in PB[c0 + b]['stations']] for h in range(HORIZON)] for b in range(B)])).mean()
    print(f'  windows {c0 + 1}-{c0 + B} of {len(widx)} | {time.time()-t0:.0f}s | mean |ensemble - playback.json| = {chk:.3f} ug/m3')
phi_all, base_all, fc_all = np.concatenate(phi_all), np.concatenate(base_all), np.concatenate(fc_all)
attn_all, band_all = np.concatenate(attn_all), np.concatenate(band_all, axis=1)

# HOW TO CHECK: the last number on each line is about 0.05 or less (rounding in playback.json). If it is above 1, the models or data
# differ from the ones used for the original export - stop and tell me.

In [ ]:
# consistency checks, then write the files
resid = fc_all - base_all - phi_all.sum(axis=2)
print('Shapley efficiency check (forecast - baseline - sum of impacts), max abs:', float(np.abs(resid).max()))
os.makedirs(os.path.join(V2_DIR, 'frontend_export'), exist_ok=True)
FE = os.path.join(V2_DIR, 'frontend_export')

ew = []
for b, w in enumerate(widx):
    stn = []
    for s in range(N):
        stn.append({'station_id': s, 'baseline': round(float(base_all[b, s]), 1), 'forecast_mean': round(float(fc_all[b, s]), 1),
                    'impacts': {gnames[g]: round(float(phi_all[b, s, g]), 1) for g in range(n_g)},
                    'attention_1e4': [int(round(x * 10000)) for x in attn_all[b, s]]})
    ew.append({'window_idx': w, 'timestamp': PB[b]['timestamp'], 'stations': stn})
json.dump({'groups': gnames, 'unit': 'ug/m3 effect on the mean 24-hour forecast; impacts + baseline = forecast_mean',
           'attention': 'weight of the last position on each of the 48 input hours, oldest first, scaled by 1e4; averaged over heads and the 7 seeds',
           'windows': ew}, open(os.path.join(FE, 'explain_windows.json'), 'w'), separators=(',', ':'))

bw = []
for b, w in enumerate(widx):
    bw.append({'window_idx': w, 'stations': [{'station_id': s, 'p10': [int(round(x)) for x in band_all[0, b, :, s]], 'p50': [int(round(x)) for x in band_all[1, b, :, s]],
                                              'p90': [int(round(x)) for x in band_all[2, b, :, s]]} for s in range(N)]})
json.dump({'definition': '10th / 50th / 90th percentile of the 7 seeds (ensemble spread, not a calibrated prediction interval)', 'windows': bw},
          open(os.path.join(FE, 'bands.json'), 'w'), separators=(',', ':'))

gl = json.load(open(os.path.join(RESULTS_DIR, 'xai_results.json'))) if os.path.exists(os.path.join(RESULTS_DIR, 'xai_results.json')) else {}
json.dump(gl, open(os.path.join(FE, 'explain_global.json'), 'w'), indent=2)

for f in ('explain_windows.json', 'bands.json', 'explain_global.json'):
    print(f'{f:24} {os.path.getsize(os.path.join(FE, f)) / 1e6:6.1f} MB')

# window-averaged impacts as a quick plausibility view
avg = phi_all.mean(axis=(0, 1))
print()
print('mean impact on the 24 h forecast, across windows and stations (ug/m3):')
for g, a in sorted(zip(gnames, avg), key=lambda t: -abs(t[1])): print(f'  {g:18} {a:+7.1f}')
print(f'  baseline {base_all.mean():.1f} -> forecast {fc_all.mean():.1f}')
print('band width (p90 - p10), mean over horizon h=+1,+6,+12,+24:', [round(float((band_all[2, :, h, :] - band_all[0, :, h, :]).mean()), 1) for h in (0, 5, 11, 23)])
print('written to', FE)

# HOW TO CHECK: efficiency check is about 0 (below 0.05). Files total about 20-30 MB for all windows. The impact table should be dominated
# by PM2.5 history and weather, with wind small - the same ranking as notebook 22.